# OMI LFCC, Swin and logistic regression — training and manuscript notebook

This notebook documents the completed **patient-grouped 1:1 undersampling study** and compares its saved results with the matched oversampled control. It follows cohort → preprocessing → models → training → calibration → validation → uncertainty. By default it verifies and uses saved artifacts without retraining. Set `OMI_NOTEBOOK_TRAIN=1` before starting the kernel to fit the full logistic and three-seed Swin procedure into a new isolated output directory. Training is research-only; official test waveforms are not decoded.

## 1. Environment, paths and verified saved study

In [1]:
import os, sys, json, hashlib, time
from pathlib import Path
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
here = Path.cwd().resolve()
REPO = next(p for p in (here, *here.parents) if (p/'experiments/acs_omi_vmd_wst_vqc/undersampled_v1/PROTOCOL.md').exists())
ATT=REPO/'attention method'
sys.path[:0]=[str(REPO),str(ATT),str(ATT/'.venv/torch-cu128')]
import numpy as np, pandas as pd, torch
from IPython.display import display
from experiments.acs_omi_vmd_wst_vqc.omi_improvement_v2 import common as c, engine as e
from experiments.acs_omi_vmd_wst_vqc.sensitivity_study_v1.common import lock
from experiments.acs_omi_vmd_wst_vqc.sensitivity_study_v1.metrics import Curve
from experiments.acs_omi_vmd_wst_vqc.reporting import binary_metrics
ACS=REPO/'experiments/acs_omi_vmd_wst_vqc'; STUDY=ACS/'undersampled_v1'; SOURCE=STUDY/'results'
assert c.verified(SOURCE)
manifest, parent_context=c.manifest()
assert c.verified(c.RUN/'prepared',parent_context)
print('Repository:',REPO,'| Python:',sys.version.split()[0],'| PyTorch:',torch.__version__)
print('Completed undersampling study and parent LFCC preparation: verified.')
print('CUDA available:',torch.cuda.is_available())

Repository: /mnt/d/Projects/test-ecg-training | Python: 3.12.3 | PyTorch: 2.7.1+cu128
Completed undersampling study and parent LFCC preparation: verified.
CUDA available: True


## 2. Cohort and patient split
The 1:1 fit subset retains all ECGs for selected subfit patients, including mixed-label patients. Calibration and validation retain natural prevalence; patient groups never overlap.

In [2]:
data=c.cohort(); selected=c.arrays(SOURCE/'cohort.npz')['indices']
cal=np.flatnonzero(data['partitions']=='calibration'); val=np.flatnonzero(data['partitions']=='validation')
assert not (set(data['patients'][selected]) & (set(data['patients'][cal])|set(data['patients'][val])))
assert not (set(data['patients'][cal]) & set(data['patients'][val]))
rows=[]
for name,ix in [('selected_train',selected),('calibration',cal),('validation',val)]:
 rows.append(dict(partition=name,ECGs=len(ix),patients=len(set(data['patients'][ix])),negative=int((data['y'][ix]==0).sum()),positive=int(data['y'][ix].sum()),positive_percent=100*float(data['y'][ix].mean())))
display(pd.DataFrame(rows));print('Patient overlap: 0. Official test waveforms were not processed.')

,partition,ECGs,patients,negative,positive,positive_percent
0,selected_train,1466,1345,733,733,50.000000
1,calibration,2866,2717,2682,184,6.420098
2,validation,3581,3391,3352,229,6.394862


Patient overlap: 0. Official test waveforms were not processed.


## 3. LFCC features and controls
Each native 10-second, 12-lead ECG is represented by 37 frames × 12 leads × 20 LFCCs. The logistic control pools means and standard deviations over time. Scaling and neural normalization use only selected training ECGs.

In [3]:
features=np.load(c.RUN/'prepared/lfcc.npy',mmap_mode='r'); pooled=np.load(c.RUN/'prepared/pooled.npy',mmap_mode='r')
protocol=manifest['protocol']; cfg=protocol['training']
print('LFCC cache shape:',features.shape,'| pooled logistic features:',pooled.shape)
print('LFCC: 500 Hz; 500-sample Hann frames; hop 125; FFT 512; 40 linear bands; 20 coefficients.')
print('Training settings:',json.dumps(cfg,indent=2))

LFCC cache shape: (17905, 37, 12, 20) | pooled logistic features: (17905, 480)
LFCC: 500 Hz; 500-sample Hann frames; hop 125; FFT 512; 40 linear bands; 20 coefficients.
Training settings: {
  "epochs": 40,
  "batch_size": 128,
  "lr": 0.001,
  "min_lr": 0.0001,
  "weight_decay": 0.01,
  "gradient_clip_norm": 1.0,
  "shuffle_seed_offset": 10000,
  "optimizer": "AdamW",
  "schedule": "cosine horizon 40, including earlier stopped refits",
  "precision": "float32",
  "deterministic": true,
  "tf32": false
}


## 4. Swin and logistic architectures
The Swin processes the 12-lead LFCC sequence; the logistic model uses standardized pooled LFCC features. These models are evaluated on the same held-out patients.

In [4]:
net=e.model('swin0',protocol)
print('Swin parameters:',sum(p.numel() for p in net.parameters()))
print('Swin configuration:',json.dumps(protocol['swin_model'],indent=2))
print('Logistic: StandardScaler fitted on unique training ECGs, C=1, lbfgs, 2,000 maximum iterations.')

Swin parameters: 2175718
Swin configuration: {
  "n_leads": 12,
  "n_ceps": 20,
  "num_classes": 2,
  "embed_dim": 64,
  "depths": [
    2,
    2,
    2
  ],
  "heads": [
    2,
    4,
    8
  ],
  "window_size": 8
}
Logistic: StandardScaler fitted on unique training ECGs, C=1, lbfgs, 2,000 maximum iterations.


## 5. Training entry point

The optional branch executes the same `fit_logistic` and `fit_neural` functions as the completed undersampling study: one logistic fit and Swin seeds 0/1/2 for 40 epochs. It uses a separate notebook-specific output folder, frozen manifest and lock. Calibration and validation remain held out. The default verifies the four completed fits and performs no new training.

In [5]:
folders={'logistic':SOURCE/'logistic', **{f'swin_seed{s}':SOURCE/f'swin_seed{s}' for s in (0,1,2)}}
for folder in folders.values(): assert c.verified(folder)
print('Verified completed logistic and all three Swin fits; selected training ECGs:',len(selected))
if os.environ.get('OMI_NOTEBOOK_TRAIN') == '1':
 device='cuda' if torch.cuda.is_available() else 'cpu'
 env=e.setup(device)
 out=ACS/'notebooks/results/notebook_retraining_v1'
 context={'parent_manifest_sha256':c.sha(c.RUN/'manifest.json'), 'selection_sha256':c.sha(SOURCE/'cohort.npz'),
          'engine_sha256':c.sha(Path(e.__file__)), 'model_sha256':c.sha(ACS/'omi_improvement_v2/models.py'),
          'protocol_sha256':c.sha(STUDY/'PROTOCOL.md'), 'environment':env, 'seeds':[0,1,2]}
 with lock(out):
  c.frozen_write(out/'manifest.json',context); ctx=c.sha(out/'manifest.json')
  if c.verified(out,ctx):
   print('Existing notebook retraining verified and reused.')
  else:
   e.fit_logistic(out/'logistic',data,pooled,selected,cal,1.,ctx)
   for seed in (0,1,2): e.fit_neural(out/f'swin_seed{seed}',data,features,selected,cal,'swin0',seed,40,protocol,ctx,device)
   import joblib
   metrics={}
   for name in ('logistic','swin_seed0','swin_seed1','swin_seed2'):
    folder=out/name
    if name=='logistic':
     scaler,clf=joblib.load(folder/'model.joblib'); cal_score=c.arrays(folder/'held.npz')['score']; val_score=clf.predict_proba(scaler.transform(pooled[val]))[:,1]
    else:
     norm=c.arrays(folder/'normalization.npz'); model=e.model('swin0',protocol).to(device)
     model.load_state_dict(torch.load(folder/'epoch40/weights.pt',map_location=device,weights_only=True))
     cal_score=c.arrays(folder/'epoch40/held.npz')['score']; val_score=e.predict(model,features,val,norm['mean'],norm['scale'],cfg['batch_size'],device)
    points=Curve(data['y'][cal],cal_score).operating_points([80,90,95]); c.frozen_write(folder/'calibration_thresholds.json',points)
    e.predictions(folder/'validation_predictions.npz',data,val,val_score);metrics[name]=binary_metrics(data['y'][val],val_score)
   c.write(out/'metrics.json',metrics);c.complete(out,ctx)
   print('Fresh fits, calibration thresholds and validation predictions saved at:',out)
else:
 print('Using saved fits. Set OMI_NOTEBOOK_TRAIN=1 before kernel start to train new models.')

Verified completed logistic and all three Swin fits; selected training ECGs: 1466
Using saved fits. Set OMI_NOTEBOOK_TRAIN=1 before kernel start to train new models.


## 6. Recompute validation metrics from saved scores
Average precision summarizes ranking across thresholds. Precision, sensitivity and specificity below use a fixed cutoff of 0.5. Swin seed results are not a seed ensemble.

In [6]:
from sklearn.metrics import average_precision_score
analysis=c.read(STUDY/'analysis.json'); validation_y=data['y'][val]
rows=[]
for name,folder in folders.items():
 z=c.arrays(folder/'validation.npz');np.testing.assert_array_equal(z['record_ids'],data['record_ids'][val]);m=binary_metrics(validation_y,z['score'])
 key='undersampled_'+name
 for k in ('average_precision','sensitivity','specificity','precision','tp','fp','fn','tn'): np.testing.assert_allclose(m[k],analysis['metrics'][key][k])
 rows.append(dict(model=key,AP=m['average_precision'],precision=m['precision'],sensitivity=m['sensitivity'],specificity=m['specificity'],TP=m['tp'],FP=m['fp']))
display(pd.DataFrame(rows).round(4));print('Validation patients:',len(set(data['patients'][val])), '| ECGs:',len(val))

,model,AP,precision,sensitivity,specificity,TP,FP
0,undersampled_logistic,0.1306,0.1142,0.6157,0.6736,141.0,1094.0
1,undersampled_swin_seed0,0.1693,0.1480,0.7293,0.7133,167.0,961.0
2,undersampled_swin_seed1,0.1435,0.1406,0.7293,0.6954,167.0,1021.0
3,undersampled_swin_seed2,0.1379,0.1211,0.6638,0.6709,152.0,1103.0


Validation patients: 3391 | ECGs: 3581


## 7. Calibration thresholds and comparison with oversampling
Each target uses a cutoff chosen on separate calibration patients. A 95% calibration target is not a guarantee of 95% sensitivity on validation. The table uses mean metrics across seeds.

In [7]:
threshold_study=ACS/'threshold_comparison_v1'; comparison=c.read(threshold_study/'analysis.json')
rows=[]
for group,values in comparison['results'].items():
 for label,v in zip(comparison['threshold_labels'],values):
  p=v['point']; rows.append(dict(model=group,cutoff=label,sensitivity=p['sensitivity'],specificity=p['specificity'],precision=p['precision'],false_positives=p['fp']))
display(pd.DataFrame(rows).round(4))
print('Threshold study bootstrap:',comparison['bootstrap'])

,model,cutoff,sensitivity,specificity,precision,false_positives
0,undersampled_logistic,fixed_0.5,0.6157,0.6736,0.1142,1094.0000
1,undersampled_logistic,cal_target_80,0.7293,0.5576,0.1012,1483.0000
2,undersampled_logistic,cal_target_90,0.8646,0.4227,0.0928,1935.0000
3,undersampled_logistic,cal_target_95,0.9083,0.3019,0.0816,2340.0000
4,undersampled_swin,fixed_0.5,0.7074,0.6932,0.1366,1028.3333
5,undersampled_swin,cal_target_80,0.7453,0.6504,0.1276,1172.0000
6,undersampled_swin,cal_target_90,0.8690,0.4406,0.0971,1875.0000
7,undersampled_swin,cal_target_95,0.9345,0.2471,0.0792,2523.6667
8,oversampled_logistic,fixed_0.5,0.5939,0.7974,0.1669,679.0000
9,oversampled_logistic,cal_target_80,0.7336,0.6751,0.1337,1089.0000


Threshold study bootstrap: {'requested': 2000, 'valid': 2000, 'seed': 20261101}


## 8. Uncertainty, saved artifacts and interpretation
The primary comparison is undersampled minus oversampled logistic AP, with a paired validation-patient bootstrap. These intervals condition on fitted models and the single selected training subset; this repeatedly studied validation set is not external validation.

In [8]:
print('Logistic AP difference:',round(analysis['primary_ap_difference'],4),'| paired 95% CI:',analysis['primary_ci'])
print('Original audit:',c.read(STUDY/'audit.json'))
print('Threshold audit:',c.read(threshold_study/'audit.json'))
print('Saved reports:',STUDY/'REPORT.md',threshold_study/'REPORT.md')

Logistic AP difference: -0.0535 | paired 95% CI: [-0.08329788845744261, -0.030886683596439698]
Original audit: {'status': 'passed', 'records': 1466, 'patients': 1345, 'seed_models_checked': 3, 'primary_metric_reproduced': True, 'calibration_thresholds_recomputed': 12, 'subfit_patients_whole': True, 'validation_patients_disjoint': True, 'manifest_sha256': 'ebd42aac26f86c1a18d59592302147aa75f8187004da79938454c3b0036ca1fa', 'results_sha256': '7e4ad00028a9bb9bd980f1da7dc219015d9d927d878abcc515d1444a337faf85'}
Threshold audit: {'status': 'passed', 'operating_points': 16, 'metrics_per_point': 7, 'bootstrap_draws': 2000, 'comparisons_note': 'individual fixed thresholds, seeds averaged; no new models or calibration fitted'}
Saved reports: /mnt/d/Projects/test-ecg-training/experiments/acs_omi_vmd_wst_vqc/undersampled_v1/REPORT.md /mnt/d/Projects/test-ecg-training/experiments/acs_omi_vmd_wst_vqc/threshold_comparison_v1/REPORT.md


No clinical validity is established. Accuracy is misleading with rare positives; assess sensitivity, precision, specificity and false-positive counts together. A threshold is chosen on calibration patients, not the validation cohort. Training, calibration and validation do not decode official test waveforms.